In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULT_DIRS = [
    Path('../../results'),
]

COLORS = {
    'Baseline': '#0072B2',
    'SGX': '#D55E00',
    'Confidential5G': '#009E73',
}

LABELS = {
    'Baseline': 'Baseline',
    'SGX': 'C5G-SGX',
    'Confidential5G': 'C5G-TDX',
}

COL_W = 3.33
TEXT_W = 7.16
FONT = {'tick': 10, 'label': 10, 'title': 10, 'legend': 8}

HATCHES = {'Baseline': '', 'SGX': '//', 'Confidential5G': 'xx'}

FIG_H = 2.4

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'Bitstream Vera Serif'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'hatch.linewidth': 0.6,
})

In [ ]:
RD = next((p for p in RESULT_DIRS if p.exists()), RESULT_DIRS[0])
IMAGES = Path('images')
IMAGES.mkdir(parents=True, exist_ok=True)

# Filenames
FILES = {
    'baseline': {
        1:   [RD / 'up_scalability_baseline_ping_1.csv'],
        10:  [RD / 'up_scalability_baseline_ping_10.csv',],
        20:  [RD / 'up_scalability_baseline_ping_20.csv',],
        50:  [RD / 'up_scalability_baseline_ping_50.csv',],
        100: [RD / 'up_scalability_baseline_ping_100.csv'],
        200: [RD / 'up_scalability_baseline_ping_200.csv',],
    },
    'sgx': {
        1:   [RD / 'up_scalability_sgx_ping_1.csv'],
        10:  [RD / 'up_scalability_sgx_ping_10.csv'],
        20:  [RD / 'up_scalability_sgx_ping_20.csv'],
        50:  [RD / 'up_scalability_sgx_ping_50.csv'],
        100: [RD / 'up_scalability_sgx_ping_100.csv'],
        200: [RD / 'up_scalability_sgx_ping_200.csv'],
    },
    'hybrid': {
        1:   [RD / 'up_scalability_hybrid_ping_1.csv'],
        10:  [RD / 'up_scalability_hybrid_ping_10.csv'],
        20:  [RD / 'up_scalability_hybrid_ping_20.csv'],
        50:  [RD / 'up_scalability_hybrid_ping_50.csv'],
        100: [RD / 'up_scalability_hybrid_ping_100.csv'],
        200: [RD / 'up_scalability_hybrid_ping_200_2.csv'],
    },
}

DEPLOYMENTS = {
    'baseline': 'Baseline',
    'hybrid':   'Confidential5G',
    'sgx':      'SGX',
}

def build_ping_df(dep_files):
    rows = []
    for n, paths in dep_files.items():
        dfs = [pd.read_csv(p) for p in paths if p.exists()]
        if not dfs:
            continue
        combined = pd.concat(dfs, ignore_index=True)
        rows.append({
            'concurrency':   n,
            'mean_rtt_ms':   combined['mean_rtt_ms'].mean(),
            'std_rtt_ms':    combined['mean_rtt_ms'].std(),
            'p95_rtt_ms':    combined['p95_ue_rtt_ms'].mean(),
            'std_p95_ms':    combined['p95_ue_rtt_ms'].std(),
            'mean_loss_pct': combined['mean_loss_pct'].mean(),
            'std_loss_pct':  combined['mean_loss_pct'].std(),
            'median_rtt_ms': combined['mean_rtt_ms'].median(),
            'q1_rtt_ms':     combined['mean_rtt_ms'].quantile(0.25),
            'q3_rtt_ms':     combined['mean_rtt_ms'].quantile(0.75),
        })
    return pd.DataFrame(rows).sort_values('concurrency').reset_index(drop=True)

ping_data = {dep: build_ping_df(FILES[dep]) for dep in DEPLOYMENTS}

print(f'Using result directory: {RD.resolve()}')
for dep, df in ping_data.items():
    print(f"{DEPLOYMENTS[dep]}")
    if df.empty:
        print('No matching files found for this deployment.')
    else:
        print(df[['concurrency','median_rtt_ms','mean_rtt_ms','p95_rtt_ms','mean_loss_pct']].to_string(index=False))
    print()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(COL_W, 5.0), gridspec_kw={'hspace': 0.85})

MARKERS = {'baseline': 'o', 'sgx': 's', 'hybrid': '^'}
X_LEVELS = [1, 10, 20, 50, 100, 200]
X_POS    = {n: i for i, n in enumerate(X_LEVELS)}

# (a) Latency
ax = axes[0]
for dep, label in DEPLOYMENTS.items():
    df = ping_data[dep]
    if df.empty: continue
    xs = [X_POS[n] for n in df['concurrency']]
    y = df['mean_rtt_ms'].values
    err = df['std_rtt_ms'].values
    ax.plot(xs, y, color=COLORS[label], marker=MARKERS[dep],
            linewidth=1.8, markersize=5, label=LABELS[label], zorder=3)
    ax.fill_between(xs, y - err, y + err, color=COLORS[label], alpha=0.25, zorder=2)

ax.set_xticks(range(len(X_LEVELS)))
ax.set_xticklabels(X_LEVELS, fontsize=FONT['tick'])
ax.tick_params(axis='y', labelsize=FONT['tick'])
ax.set_xlabel('Concurrent UEs', fontsize=FONT['label'])
ax.set_ylabel('Mean RTT (ms)', fontsize=FONT['label'])
ax.set_title('(a) Latency', fontsize=FONT['title'], pad=6)
ax.legend(fontsize=FONT['legend'], frameon=False, loc='upper left')
ax.grid(axis='y', linestyle='-', alpha=0.15)
ax.set_ylim(0, 18)

# (b) Packet Loss
ax = axes[1]
for dep, label in DEPLOYMENTS.items():
    df = ping_data[dep]
    if df.empty: continue
    xs = [X_POS[n] for n in df['concurrency']]
    y = df['mean_loss_pct'].values
    err = df['std_loss_pct'].values
    ax.plot(xs, y, color=COLORS[label], marker=MARKERS[dep],
            linewidth=1.8, markersize=5, label=LABELS[label], zorder=3)
    ax.fill_between(xs, np.maximum(y - err, 0), y + err, color=COLORS[label], alpha=0.25, zorder=2)

ax.set_xticks(range(len(X_LEVELS)))
ax.set_xticklabels(X_LEVELS, fontsize=FONT['tick'])
ax.tick_params(axis='y', labelsize=FONT['tick'])
ax.set_xlabel('Concurrent UEs', fontsize=FONT['label'])
ax.set_ylabel('Packet Loss (%)', fontsize=FONT['label'])
ax.set_title('(b) Packet Loss', fontsize=FONT['title'], pad=6)
ax.legend(fontsize=FONT['legend'], frameon=False, loc='upper left')
ax.grid(axis='y', linestyle='-', alpha=0.15)
ax.set_ylim(0, 110)

fig.tight_layout()
plt.savefig(IMAGES / 'up_scalability_ping.png', bbox_inches='tight', dpi=300)
plt.savefig(IMAGES / 'up_scalability_ping.pdf', bbox_inches='tight')
plt.show()